# 🧠 RAG completo con Gemini y LangChain

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque 5 — 50 minutos**

---

## Objetivo

Construir un sistema RAG end-to-end: documentos → fragmentos → embeddings → ChromaDB → Gemini → respuesta fundamentada.

## Al terminar este bloque vas a poder:

1. Conectar ChromaDB como retriever con Gemini como generador usando LangChain.
2. Diseñar prompts RAG que fuercen al modelo a responder solo con el contexto.
3. Agregar documentos nuevos a un sistema RAG existente sin reiniciarlo.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **RAG completo** | Pipeline que une retrieval (ChromaDB) con generation (LLM). |
| **Retriever** | Componente que recibe una pregunta y devuelve los K chunks más relevantes. |
| **Context budget** | El límite de tokens que podemos inyectar en el prompt sin degradar al modelo. |
| **RetrievalQA** | Clase de LangChain que orquesta el pipeline RAG de principio a fin. |
| **MMR (Max Marginal Relevance)** | Estrategia de retrieval que evita chunks repetitivos entre sí. |

In [ ]:
!uv pip install websockets

In [7]:
# Instalamos las librerías necesarias para nuestro sistema RAG
# LangChain: orquestador del pipeline RAG
# langchain-google-genai: conector de LangChain con la API de Gemini
# langchain-chroma: conector de LangChain con ChromaDB
# sentence-transformers: embeddings locales sin consumir cuota de API
!uv pip install langchain langchain-google-genai langchain-chroma langchain-community chromadb sentence-transformers -q

print("✓ Todas las librerías instaladas correctamente")
print("  IMPORTANTE: Solo se usará API de Gemini para generación final de respuestas")

✓ Todas las librerías instaladas correctamente
  IMPORTANTE: Solo se usará API de Gemini para generación final de respuestas


SyntaxError: invalid syntax (391436018.py, line 2)

In [53]:
from langchain_community.llms import Ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader # o TextLoader para txt

from langchain_classic.chains.retrieval_qa.base import RetrievalQA

# Conectamos con el modelo Phi-3 que está corriendo en Ollama
llm = Ollama(model="gemma4:2b")

# Usamos un modelo de embeddings ultra liviano que corre en CPU sin problemas
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Modelos cargados correctamente.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6214.77it/s]


Modelos cargados correctamente.


In [54]:
import os
import warnings
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.embeddings import SentenceTransformerEmbeddings

load_dotenv()
warnings.filterwarnings("ignore")

print("✓ Librerías cargadas exitosamente")
print("  Usando embeddings locales para reducir uso de API")

✓ Librerías cargadas exitosamente
  Usando embeddings locales para reducir uso de API


## El sistema completo: ensamblando las piezas

### Analogía

Hasta acá tenías dos herramientas por separado: el buscador (ChromaDB) y el redactor (Gemini). RAG es el proceso que las conecta: el buscador encuentra los tres párrafos más relevantes, los pone frente al redactor, y el redactor escribe la respuesta mirando solo esos párrafos.

### Dónde vive esto en el mundo real

Este es el mismo patrón que usan los asistentes de documentos de Notion AI y los chatbots bancarios. El LLM solo puede responder con lo que está en tus documentos — eso hace las respuestas verificables y evita alucinaciones sobre tu dominio específico.

### Flujo del sistema

```
[Pregunta]  →  ChromaDB.query(k=3)  →  [3 chunks relevantes]
                                               |
                       Gemini  ←  [prompt: contexto + pregunta]
                                               |
                           [Respuesta fundamentada + fuentes]
```

### ✎ Para pensar

- ¿Por qué el prompt tiene la instrucción "si no encontrás la información, decí que no sabés"?
- Si aumentás k de 3 a 10 chunks, ¿qué ventajas y qué problemas puede traer?

## Paso 1 — Documentos y fragmentación

El `RecursiveCharacterTextSplitter` divide por párrafo primero, luego por oración, luego por espacio — respetando coherencia semántica.

### ◈ Nota: documentos de ejemplo vs documentos reales

En este notebook usamos documentos hardcodeados sobre cultura argentina para que puedas ejecutar el sistema sin dependencias externas.

Cuando dispongas de tu PDF, reemplazá la variable `documentos_langchain` por el resultado de cargar el archivo con `PyPDFLoader` (lo viste en el notebook anterior):

```python
# Cuando tengas el PDF disponible:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("ruta/a/tu/documento.pdf")
documentos_langchain = loader.load()
```

Por ahora el sistema funciona igual — la lógica de RAG no cambia según la fuente.

In [56]:
# 1. Cargar el documento
loader = PyPDFLoader("CARPETA_DATOS/OPD2.pdf")
loader1 = PyPDFLoader("CARPETA_DATOS/PERSONALIDAD.pdf")
loader2 = PyPDFLoader("CARPETA_DATOS/OPD3.pdf")
loader3 = PyPDFLoader("CARPETA_DATOS/Freud.pdf")
documentos = loader.load()
documentos.extend(loader1.load())
documentos.extend(loader2.load())
documentos.extend(loader3.load())

# 2. Dividir el texto en fragmentos (chunks) más pequeños
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
textos_divididos = text_splitter.split_documents(documentos)

print(f"Documento dividido en {len(textos_divididos)} fragmentos.")

parsing for Object Streams


Documento dividido en 4963 fragmentos.


In [29]:
# 3. Crear la base de datos vectorial con los fragmentos y los embeddings
# Esto guardará la base de datos en una carpeta local llamada "chroma_db"
vectorstore = Chroma.from_documents(
    documents=textos_divididos, 
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# Creamos el recuperador (retriever)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Recupera los 3 fragmentos más relevantes

print("Base de datos vectorial creada y lista para consultar.")

Base de datos vectorial creada y lista para consultar.


In [33]:
# 4. Crear la cadena de Pregunta-Respuesta (QA)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

# 5. ¡Hacer la consulta!
pregunta = "Me podrias explicar qué es el OPD y cuáles son sus aplicaciones clínicas?"

respuesta = qa_chain.invoke(pregunta)

print("📝 RESPUESTA DEL SISTEMA RAG:\n")
print(respuesta['result'])

📝 RESPUESTA DEL SISTEMA RAG:

El Enfoque Psicosomático (OPD) es una aproximación terapéutica que aborda los trastornos psicológicos no solo como procesos mentales internos, sino también en relación con la salud somática del individuo. Este enfoque integra y considera el bienestar físico dentro de su tratamiento psicológico para mejorar tanto mental como corporalmente los pacientes que se presentan ante las clínicas psicosomáticas.

En cuanto a sus aplicaciones clínicas, un aspecto clave es la identificación y evaluación del trastorno mediante diferentes formas de medir el cambio dentro de cada uno de los ejes biopsicológicos que aborda (físico-biológico, somático, afectivo-expresión emocional e interpersonal). La escala utilizada para esta evaluación se basa en la teoría del Diseño Psicosomático y su capacidad de cambiar refuerza el progreso hacia una mejor salud.

La OPD tiene como objetivo no solo tratar los síntomas sino también abordar las causas subyacentes que puedan estar relaci

In [34]:
# Creamos documentos de ejemplo sobre cultura y entretenimiento
# En la práctica, estos datos vendrían de archivos PDF, Word, bases de datos, etc.
documentos_cultura = [
    {
        "titulo": "Guia de Cine Argentino Contemporaneo",
        "contenido": """
        Directores Destacados: Lucrecia Martel es considerada una de las directoras mas importantes de Latinoamerica.
        Sus peliculas como La Cienaga (2001) y Zama (2017) recibieron reconocimiento internacional.
        Damian Szifron gano el Goya a Mejor Pelicula Extranjera con Relatos Salvajes (2014).

        Festivales y Premios: El Festival de Mar del Plata es el unico festival de cine Clase A en Latinoamerica, reconocido por FIAPF.
        El BAFICI (Buenos Aires Festival Internacional de Cine Independiente) se realiza cada abril desde 1999.

        Nuevos Talentos: El cine argentino reciente destaca por peliculas como Argentina, 1985 (2022) que gano el Globo de Oro
        y fue nominada al Oscar. Directoras como Paula Hernandez y Celina Murga estan renovando el panorama cinematografico.
        """
    },
    {
        "titulo": "Rock Nacional: Historia y Figuras Clave",
        "contenido": """
        Origenes: El rock argentino comenzo en los 60 con bandas como Los Gatos y Almendra.
        Luis Alberto Spinetta es considerado el poeta del rock nacional. Murio en 2012 dejando un legado de mas de 40 años de carrera.

        Era Dorada: Los 80 vieron el auge con Soda Stereo, Patricio Rey y sus Redonditos de Ricota, Los Fabulosos Cadillacs.
        Gustavo Cerati es reconocido como uno de los musicos mas influyentes del rock en español.

        Actualidad: Bandas como Estelares, Las Pastillas del Abuelo, y la escena indie con El Mato un Policia Motorizado
        mantienen vigente el rock argentino. Los festivales como Cosquin Rock y Lollapalooza Argentina reunen multitudes cada año.
        """
    },
    {
        "titulo": "Videojuegos y Cultura Gamer en Argentina",
        "contenido": """
        Desarrollo Local: Argentina tiene una industria de videojuegos en crecimiento. Estudios como NGD Studios (Maestros del Mana)
        y Three Headed Monkey produjeron titulos que compitieron internacionalmente.

        Esports y Comunidad: Argentina cuenta con jugadores profesionales destacados en League of Legends, Counter-Strike y Dota 2.
        El Tecnopolis Gaming fue uno de los eventos mas grandes de gaming en Latinoamerica antes de la pandemia.

        Cultura Popular: Los streamers argentinos tienen millones de seguidores. La comunidad gamer se reune en eventos como
        Argentina Game Show (AGS), el evento gamer mas grande del pais, que se realiza anualmente en Buenos Aires.
        Juegos como Free Fire y FIFA son extremadamente populares entre jovenes argentinos.
        """
    },
    {
        "titulo": "Teatro y Cultura Porteña",
        "contenido": """
        Corriente Teatral: La Avenida Corrientes es conocida como la Broadway porteña. Concentra mas de 30 teatros
        y recibe 5 millones de espectadores anuales. El Teatro Colon es uno de los mejores teatros de opera del mundo
        por su acustica perfecta.

        Teatro Independiente: Buenos Aires tiene mas de 200 salas de teatro independiente. El barrio de Villa Crespo
        concentra muchas salas off, donde se experimenta con dramaturgia contemporanea.

        Festivales: El Festival Internacional de Buenos Aires (FIBA) se realiza bienalmente y trae compañias de todo el mundo.
        El Teatro San Martin ofrece funciones gratuitas y es referente de teatro nacional de calidad.
        """
    }
]

print(f"✓ {len(documentos_cultura)} documentos de cultura argentina listos:")
for i, doc in enumerate(documentos_cultura, 1):
    print(f"   {i}. {doc['titulo']}")

✓ 4 documentos de cultura argentina listos:
   1. Guia de Cine Argentino Contemporaneo
   2. Rock Nacional: Historia y Figuras Clave
   3. Videojuegos y Cultura Gamer en Argentina
   4. Teatro y Cultura Porteña


In [35]:
# El "Text Splitter" divide documentos grandes en secciones manejables
# manteniendo el contexto con chunk_overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # cada fragmento tendrá máximo 500 caracteres
    chunk_overlap=50,     # 50 caracteres se superponen entre fragmentos para mantener contexto
    separators=["\n\n", "\n", ".", " "]
)

# Convertimos nuestros documentos al formato que entiende LangChain
documentos_langchain = []
for doc in documentos_cultura:
    documento = Document(
        page_content=doc["contenido"],
        metadata={"titulo": doc["titulo"]}
    )
    documentos_langchain.append(documento)

# Dividimos todos los documentos en fragmentos más pequeños
fragmentos = text_splitter.split_documents(documentos_langchain)

print(f"✦ Documentos originales: {len(documentos_cultura)}")
print(f"✦ Fragmentos generados: {len(fragmentos)}")
print(f"\n◈ Ejemplo de fragmento:")
print(f"  Título: {fragmentos[0].metadata['titulo']}")
print(f"  Contenido: {fragmentos[0].page_content[:200]}...")

✦ Documentos originales: 4
✦ Fragmentos generados: 9

◈ Ejemplo de fragmento:
  Título: Guia de Cine Argentino Contemporaneo
  Contenido: Directores Destacados: Lucrecia Martel es considerada una de las directoras mas importantes de Latinoamerica.
        Sus peliculas como La Cienaga (2001) y Zama (2017) recibieron reconocimiento inter...


In [36]:
print(f"Título: {fragmentos[2].metadata['titulo']}")
print(f"Contenido: {fragmentos[2].page_content[:200]}...")

Título: Guia de Cine Argentino Contemporaneo
Contenido: Nuevos Talentos: El cine argentino reciente destaca por peliculas como Argentina, 1985 (2022) que gano el Globo de Oro
        y fue nominada al Oscar. Directoras como Paula Hernandez y Celina Murga e...


## Paso 2 — Embeddings locales y ChromaDB

`multilingual-e5-large` corre localmente (sin API). Gemini solo se usa para la generación final — ahorrás 70-80% de cuota.

In [ ]:
# Configuramos embeddings locales multilenguaje
# Este modelo convierte texto en vectores y entiende bien el español rioplatense
embeddings = SentenceTransformerEmbeddings(
    model_name="intfloat/multilingual-e5-large"
)

print("✓ Modelo de embeddings local configurado (multilingual-e5-large)")
print("  No consume cuota de API — todo el procesamiento es local")

In [37]:
# ChromaDB guarda los vectores y permite búsqueda por similitud semántica
# LangChain se encarga de vectorizar los fragmentos y cargarlos en ChromaDB
vectorstore = Chroma.from_documents(
    documents=fragmentos,
    embedding=embeddings,
    collection_name="documentos_cultura",
    persist_directory="./chroma_db"
)

print(f"✓ Base de conocimiento vectorial creada con {len(fragmentos)} fragmentos")
print("  Embeddings procesados localmente — sin consumir cuota de Gemini")

✓ Base de conocimiento vectorial creada con 9 fragmentos
  Embeddings procesados localmente — sin consumir cuota de Gemini


## Paso 3 — Configuración de Gemini

`temperature=0.1` para respuestas precisas y consistentes.

In [ ]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "").strip()

if not GOOGLE_API_KEY:
    # Alternativa: usar GEMINI_API_KEY si GOOGLE_API_KEY no está configurada
    GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()

if not GOOGLE_API_KEY:
    raise ValueError(
        "Falta configurar GOOGLE_API_KEY (o GEMINI_API_KEY) en el archivo .env\n"
        "Obtené tu key gratuita en: https://aistudio.google.com/apikey"
    )

# Configuramos el modelo Gemini que generará las respuestas finales
# NOTA: Solo este componente consume cuota de API, los embeddings son locales
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,
    google_api_key=GOOGLE_API_KEY
)

print("✓ Modelo Gemini configurado")
print("  Modelo: gemini-2.5-flash (rápido y preciso)")
print("  Temperatura: 0.1 (respuestas consistentes y factuales)")
print("  IMPORTANTE: Solo la generación final usa API de Gemini")

## Paso 4 — Prompt template

El prompt le dice al modelo: usá SOLO el contexto provisto. Si no está ahí, decí que no sabés.

In [38]:
# El prompt template define el contrato entre el sistema y el LLM
template_respuesta = """
Sos un asistente experto en cultura, entretenimiento y artes.
Tu trabajo es responder preguntas basandote UNICAMENTE en la informacion
proporcionada en los documentos culturales.

INSTRUCCIONES IMPORTANTES:
1. Solo usa informacion que aparece explicitamente en los documentos
2. Si no encontras la informacion especifica, decilo claramente
3. Cita el documento o seccion cuando sea posible
4. Se preciso con nombres, fechas y datos
5. Usa un tono amigable pero informado

CONTEXTO DE LOS DOCUMENTOS:
{context}

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:
"""

# PromptTemplate valida que los placeholders {context} y {question} estén presentes
prompt = PromptTemplate(
    template=template_respuesta,
    input_variables=["context", "question"]
)

print("✓ Template de respuesta configurado")
print("  El asistente seguirá instrucciones específicas para dar respuestas precisas")

✓ Template de respuesta configurado
  El asistente seguirá instrucciones específicas para dar respuestas precisas


## Paso 5 — Ensamblado del pipeline

`RetrievalQA` conecta ChromaDB + Gemini + prompt template. Con `return_source_documents=True` podés mostrar qué fragmentos se usaron.

In [39]:
# LCEL reemplaza a RetrievalQA (eliminado en LangChain 1.x)
# La cadena conecta: retriever → prompt → llm → parser de texto

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

sistema_rag = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✓ Sistema RAG completamente configurado")
print("\nFlujo de trabajo del sistema:")
print("   1. Usuario hace una pregunta")
print("   2. ChromaDB busca los 3 fragmentos más relevantes (LOCAL)")
print("   3. Gemini lee esos fragmentos y genera una respuesta (API)")
print("   4. Se devuelve la respuesta + documentos fuente")
print("\nVentaja: 70-80% menos uso de API de Gemini")

✓ Sistema RAG completamente configurado

Flujo de trabajo del sistema:
   1. Usuario hace una pregunta
   2. ChromaDB busca los 3 fragmentos más relevantes (LOCAL)
   3. Gemini lee esos fragmentos y genera una respuesta (API)
   4. Se devuelve la respuesta + documentos fuente

Ventaja: 70-80% menos uso de API de Gemini


## Probando el sistema

Las primeras cuatro preguntas están en los documentos. La quinta está fuera del scope — observá cómo responde el sistema.

### 🐛 Laboratorio de Rotura

El sistema que armamos tiene una instrucción clave en el prompt: *"respondé SOLO con lo que está en los documentos"*. Antes de verlo en acción, predecí: ¿qué pasaría si esa instrucción no estuviera?

Ejecutá la celda siguiente para comparar las dos versiones con la misma pregunta sobre un tema que **no está en ningún documento**.

- ¿Qué responde el sistema con restricción?
- ¿Qué responde el sistema sin restricción?
- ¿Cuál es más útil en un contexto real? ¿Por qué?

No mires la explicación todavía.

In [40]:
template_sin_restriccion = """
Sos un experto en cultura y entretenimiento. Respondé la siguiente pregunta
con toda la información que tengas disponible.

Contexto de documentos:
{context}

Pregunta: {question}

Respuesta:
"""

prompt_sin_restriccion = PromptTemplate(
    template=template_sin_restriccion,
    input_variables=["context", "question"]
)

sistema_sin_restriccion = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_sin_restriccion
    | llm
    | StrOutputParser()
)

pregunta_fuera_de_scope = "¿Quién ganó el último Grammy Latino?"

print("CON RESTRICCIÓN ('decí que no sabés si no está en el contexto'):")
print("-" * 70)
resultado_con = sistema_rag.invoke(pregunta_fuera_de_scope)
print(resultado_con)

print("\nSIN RESTRICCIÓN:")
print("-" * 70)
resultado_sin = sistema_sin_restriccion.invoke(pregunta_fuera_de_scope)
print(resultado_sin)

print("\n◈ El sistema sin restricción mezcla conocimiento del modelo con el contexto.")
print("  Eso hace las respuestas más difíciles de verificar y auditar.")

CON RESTRICCIÓN ('decí que no sabés si no está en el contexto'):
----------------------------------------------------------------------
Lo siento, no puedo proporcionar la respuesta específica ya que mi conocimiento se limita a los eventos antes de mi última actualización en abril del año pasado. Si quisieras saber el ganador reciente del Grammy Latino para películas o canciones en particular, recomendaría buscarlo directamente en la página oficialmente anfitrión por su excelente calidad y diversidad cultural que refleja las influencias latinas.

SIN RESTRICCIÓN:
----------------------------------------------------------------------
El último premio que ganó un artista argentino en los Grammys fue Shakira con su álbum "Lived on Borrowed Time" (2de enero de 2020). Sin embargo, si estamos hablando especificamente del Grammy Latino y no del general, el último ganador sería Carlos Vives por la canción "Despedida", parte del álbum “Vivir es mi derecho”, que se llevó a casa este premio en 20

In [41]:
def hacer_pregunta(pregunta, mostrar_fuentes=True):
    print(f"\n{'='*60}")
    print(f"PREGUNTA: {pregunta}")
    print(f"{'='*60}")

    respuesta = sistema_rag.invoke(pregunta)

    print(f"\nRESPUESTA DEL SISTEMA:")
    print(respuesta)

    if mostrar_fuentes:
        fuentes = retriever.invoke(pregunta)
        print(f"\nDOCUMENTOS CONSULTADOS:")
        for i, doc in enumerate(fuentes, 1):
            print(f"   {i}. {doc.metadata['titulo']}")
            print(f"      Fragmento: {doc.page_content[:100]}...")

    return respuesta

print("✓ Función de prueba lista")

✓ Función de prueba lista


In [42]:
# Preguntamos sobre cine argentino
resultado1 = hacer_pregunta("¿Quien es Lucrecia Martel?")


PREGUNTA: ¿Quien es Lucrecia Martel?

RESPUESTA DEL SISTEMA:
Lucrecia Martel, nacida el 29 de marzo de 1963 en Cienfuegos, Cuba, es una cineasta argentina muy respetada que ha contribuido significativamente al panorama cultural latinoamericano. Su obra "La Cienaga" (2001) y su reciente película "Zama" (2017), son notables por haber sido reconocidas a nivel internacional, incluyendo el Goya a Mejor Película Extranjera para Szifron en 2014. Martel es una voz clave dentro del cine latinoamericano que nos lleva reflexionar sobre las complejas realidades sociales y políticas de su entorno cultural.

Referencia: El documento señala la contribución aportada por Lucrecia Martel al cine con sus películas notables como "La Cienaga" (2001) e "Hijo de los Hermanos Somersaultes" (2013), aunque no se menciona esta última.

DOCUMENTOS CONSULTADOS:
   1. Guia de Cine Argentino Contemporaneo
      Fragmento: Directores Destacados: Lucrecia Martel es considerada una de las directoras mas importantes de

In [43]:
# Preguntamos sobre rock nacional
resultado2 = hacer_pregunta("¿Cuales fueron los hits de Soda Stereo?")


PREGUNTA: ¿Cuales fueron los hits de Soda Stereo?

RESPUESTA DEL SISTEMA:

Sin acceso directo a la lista exhaustiva de sus éxitos, no puedo proporcionar una respuesta detallada sin información explícita del documento. No obstante, soy consciente que Soda Stereo fue un grupo muy influyente en los años '80 y su música todavía es celebrada hoy día. Si deseas saber más sobre sus hits, sería recomendable consultar bases de datos musicales o discográficas especializadas que recopilan esta información detalladamente.

DOCUMENTOS CONSULTADOS:
   1. Rock Nacional: Historia y Figuras Clave
      Fragmento: Origenes: El rock argentino comenzo en los 60 con bandas como Los Gatos y Almendra.
        Luis Alb...
   2. Rock Nacional: Historia y Figuras Clave
      Fragmento: Actualidad: Bandas como Estelares, Las Pastillas del Abuelo, y la escena indie con El Mato un Polici...
   3. Teatro y Cultura Porteña
      Fragmento: Festivales: El Festival Internacional de Buenos Aires (FIBA) se realiza bien

In [44]:
# Preguntamos sobre videojuegos argentinos
resultado3 = hacer_pregunta("¿Se producen videojuegos en Argentina?")


PREGUNTA: ¿Se producen videojuegos en Argentina?

RESPUESTA DEL SISTEMA:
Sí, efectivamente se producen videojuegos en Argentina y la industria ha estado experimentando un crecimiento significativo. Estudios locales como NGD Studios (conocidos por sus trabajos con Mana), así como Three Headed Monkey, han obtenido reconocimientos internacionales destacándose incluso dentro de las mejores producciones del año, según datos provenientes de eventos y premiaciones globales. Por ejemplo:

- NGD Studios se mantiene activo en la producción local con juegos como "Fortnite", que ha sido un éxito tanto en Argentina como internacionalmente (información extraída basándose en los logros obtenidos por el estudio).

En cuanto al esports, jugadores profesionales de Argentina han ganado reconocimiento y participan activamente en competiciones significativas. El Tecnopolis Gaming, antes una plataforma líder para estos eventos dentro del continente Latinoamericano, sirvió como un testamento a la pasión arg

In [45]:
# Preguntamos sobre teatro porteño
resultado4 = hacer_pregunta("¿Cuantas salas de teatro hay en Buenos Aires?")


PREGUNTA: ¿Cuantas salas de teatro hay en Buenos Aires?

RESPUESTA DEL SISTEMA:
Buenas tardes, para asegurar la precisión al responder sobre los teatros en Buenos Aires, se contabilizan más de 200 salas dentro del circuito local. Sin embargo, si consideramos el panorama teatral argentino incluyendo escenarios alternativos y independientes como las "off" ubicadas especialmente en Villa Crespo, esa cantidad puede aumentar significativin
gamente. Los datos que se me han proporcionado son hasta marzo de 2023, por lo tanto, podrían haber habido cambios recientes después de ese punto. Para obtener información más actualizada sobre el número exacto de teatros en Buenos Aires, sería recomendable consultar fuentes como guías turísticas locales o bases de datos específicas del arte y la cultura dentro del país.

DOCUMENTOS CONSULTADOS:
   1. Teatro y Cultura Porteña
      Fragmento: Festivales: El Festival Internacional de Buenos Aires (FIBA) se realiza bienalmente y trae compañias...
   2. Tea

In [46]:
# Probamos qué pasa cuando preguntamos algo que no está en nuestros documentos
resultado5 = hacer_pregunta("¿Cuál fue la mejor película internacional del año pasado?")


PREGUNTA: ¿Cuál fue la mejor película internacional del año pasado?

RESPUESTA DEL SISTEMA:
En última instancia, determinar una "mejor" película es subjetivo y puede variar según las preferencias personales. Sin embargo, basándome en el reconocimiento reciente por parte de instituciones importantes del cine internacional como la Academia Británica con los premios Óscar, que reconoce a películas tanto nacionales como extranjeras, mi recomendación sería "Moonlight" (2016) dirigida por Barry Jenkins. Esta película ganó el Oscar a Mejor Película y fue altamente laogada por su profunda narrativa y dirección sobresaliente en temas de identidad e interacciones personales.

La obra del director argentino Lucrecia Martel, aunque extraordinaria dentro del cine latinoamericano como reconocimiento a su importancia cultural y artística, no obtuvo este tipo de galardones internacionales específicamente en el último año que pasa la pregunta. Dado que nuestra información está limitada al contexto pro

### ✎ Para pensar

- ¿Cómo manejó el sistema la pregunta fuera del scope? ¿Inventó algo o admitió que no sabía?
- Si quisieras que el sistema cite explícitamente el documento fuente en cada respuesta, ¿qué cambiarías en el prompt template?

## Actualización dinámica

Una ventaja clave de RAG: podés agregar documentos nuevos sin reiniciar ni reentrenar. El sistema los indexa y los usa en la próxima consulta.

In [49]:
# Simulamos que llegan nuevos documentos culturales
nuevos_documentos = [
    {
        "titulo": "Musica Urbana Argentina: Trap y Hip Hop",
        "contenido": """
        Referentes: Duki, Bizarrap y Nicki Nicole son algunos de los mayores exponentes del trap argentino.
        Duki lleno el estadio de Velez dos veces consecutivas en 2022, marcando un hito historico.
        Bizarrap llego a 50 millones de suscriptores en YouTube con sus Music Sessions.

        Escena Local: El trap argentino nacio en plazas y cypher callejeros. La escena se profesionalizo
        en la ultima decada con productoras y sellos discograficos dedicados al genero.

        Impacto Internacional: Artistas argentinos colaboran con figuras internacionales.
        Bizarrap produjo sesiones con Shakira, Residente y Quevedo que rompieron records de reproduccion.
        """
    },
    {
        "titulo": "Literatura Argentina Contemporanea",
        "contenido": """
        Autores Destacados: Claudia Piñeiro, Samanta Schweblin y Mariana Enriquez son referentes de la literatura
        argentina actual. Sus obras se traducen a multiples idiomas y ganan premios internacionales.

        Feria del Libro: La Feria del Libro de Buenos Aires es una de las mas importantes de habla hispana.
        Se realiza anualmente en La Rural y recibe mas de un millon de visitantes.

        Premios: Argentina cuenta con premios literarios importantes como el Clarin de Novela y el Premio Konex.
        Escritores argentinos fueron galardonados con el Premio Cervantes, maximo reconocimiento en español.
        """
    }
]

print(f"✓ {len(nuevos_documentos)} documentos nuevos listos para agregar:")
for doc in nuevos_documentos:
    print(f"   - {doc['titulo']}")

✓ 2 documentos nuevos listos para agregar:
   - Musica Urbana Argentina: Trap y Hip Hop
   - Literatura Argentina Contemporanea


In [50]:
def agregar_documentos(nuevos_docs, vs):
    """
    Agrega nuevos documentos al vectorstore existente.
    El sistema los puede consultar inmediatamente en la próxima pregunta.

    Parámetros:
    -----------
    nuevos_docs : list
        Lista de dicts con claves 'titulo' y 'contenido'
    vs : Chroma
        El vectorstore donde agregar los documentos
    """
    print("Procesando nuevos documentos...")

    docs_langchain = [
        Document(page_content=doc["contenido"], metadata={"titulo": doc["titulo"]})
        for doc in nuevos_docs
    ]

    nuevos_fragmentos = text_splitter.split_documents(docs_langchain)
    print(f"✦ Fragmentos generados: {len(nuevos_fragmentos)}")

    vs.add_documents(nuevos_fragmentos)

    print(f"✓ Documentos agregados exitosamente a la base de conocimiento")
    return len(nuevos_fragmentos)

# Agregamos los nuevos documentos
agregar_documentos(nuevos_documentos, vectorstore)

Procesando nuevos documentos...
✦ Fragmentos generados: 4
✓ Documentos agregados exitosamente a la base de conocimiento


4

In [51]:
# Probamos preguntas sobre los documentos recién agregados
print("Probando preguntas sobre los documentos nuevos:\n")

hacer_pregunta("¿Qué libros escribió Samanta Schweblin?")
hacer_pregunta("¿Quién es Bizarrap?")

Probando preguntas sobre los documentos nuevos:


PREGUNTA: ¿Qué libros escribió Samanta Schweblin?

RESPUESTA DEL SISTEMA:
Samanta Schweblin es una escritora contemporánea argentina, conocida principalmente por su novela "Sigma", que ganó el prestigioso Premio Dashwood en la región de habla hispana. Otra obra destacada de ella es "Lacetazos", un relato corto cuyas historias fueron incluídas en diversas antologías y han recibido aclamaciones críticas. Estas son algunas obras clave que abarcan su vasta producción literaria:

1. **Sigma** (2014): Esta novela de terror se considera una obra maestra en la narrativa actual latinoamericana, reflejando temas universales como el amor y el miedo a través del prisma psicológico complejo que Schweblin crea con sus personajes.
2. **Lacetazos**: Esta colección de cuentos es reconocida por su estilo único, mezclando lo cotidiano con elementos fantásticos y surrealistas.
3. endorsed works include her collection of short stories "Terraplen" (2015), ex

'Bizarrap, también conocido como Ricardo "Nacho" Repilado Fernández, es un productor y DJ de Argentina. Es más famoso por sus Music Sessions en YouTube que han logrado alcanzar la marca de 50 millones de suscriptores a nivel mundial. Sin embargo, no hay evidencia directa dentro del context proporcionado sobre su nacimiento o vida personal aparte de su trabajo en el ámbito musical y digital.\n\nCITAS DOCUMENTOS: "Bizarrap ha logrado alcanzar la marca de 50 millones de suscriptores con sus Music Sessions en YouTube, lo que le ha conferido una prominencia considerable en internet."'

In [52]:
# --- Espacio de practica ---
#
# Construí tu propio mini-RAG:
#   1. Crea 3-4 documentos sobre un tema que te interese
#   2. Agregalos con agregar_documentos(mis_docs, vectorstore)
#   3. Hacé 3 preguntas: 2 que estén en scope y 1 que no
#   4. Observá cómo el sistema usa las fuentes
#
# Bonus: modificá el template para que cite siempre el título del documento

mis_docs = [
    {"titulo": "Mi tema 1", "contenido": "Tu contenido 1"},
    {"titulo": "Mi tema 2", "contenido": "Tu contenido 2"},
]

agregar_documentos(mis_docs, vectorstore)
hacer_pregunta("Tu pregunta sobre el tema")

Procesando nuevos documentos...
✦ Fragmentos generados: 2
✓ Documentos agregados exitosamente a la base de conocimiento

PREGUNTA: Tu pregunta sobre el tema

RESPUESTA DEL SISTEMA:
La música y la cultura que hacen sonidos las bandas argentinas han tenido un impacto significativo no solo dentro del país sino en todo el continente. A fines de los años '60, con nombres como Los Gatos y Almendra liderando su propio estilo distintivo llamado "rock nacional", Argentina empezó a sonar más fuerte en la escena musical mundial.

Luis Alberto Spinetta, que nació el 25 de junio de 1953 y falleció el 8 de febrero de 2012, es considerado por muchos como uno de los mejores cantantes y compositores del rock argentino. A pesar de su muerte a la edad de 58 años, su legado musical continúa siendo una fuente inagotable de inspiración para las nuevas generaciones. Spinetta fue premiado en numerosas ocasiones por sus contribuciones al rock y dejó detrás un catálogo extenso que incluye álbumes como "Artaud" 

'La música y la cultura que hacen sonidos las bandas argentinas han tenido un impacto significativo no solo dentro del país sino en todo el continente. A fines de los años \'60, con nombres como Los Gatos y Almendra liderando su propio estilo distintivo llamado "rock nacional", Argentina empezó a sonar más fuerte en la escena musical mundial.\n\nLuis Alberto Spinetta, que nació el 25 de junio de 1953 y falleció el 8 de febrero de 2012, es considerado por muchos como uno de los mejores cantantes y compositores del rock argentino. A pesar de su muerte a la edad de 58 años, su legado musical continúa siendo una fuente inagotable de inspiración para las nuevas generaciones. Spinetta fue premiado en numerosas ocasiones por sus contribuciones al rock y dejó detrás un catálogo extenso que incluye álbumes como "Artaud" (1975) y una serie notablemente exitosa de discos desde los años \'80 hasta su muerte.\n\nDurante la década del 80, surgieron bandas más ampliamente reconocidas en Argentina tal

> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: mirá el `template_respuesta` que armamos y encontrá la instrucción que te parece más importante. Escribí, arriba de ese bloque, un comentario de dos líneas que explique por qué esa instrucción está ahí y qué pasaría sin ella. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

## Cierre del bloque y de la cursada

| Componente RAG | Qué hace |
|---|---|
| **Document Loaders** | Extraen texto de PDFs, webs, etc. |
| **Text Splitter** | Fragmenta en chunks coherentes con overlap |
| **Embeddings locales** | Vectorizan los chunks sin consumir API |
| **ChromaDB** | Almacena y busca vectores por similitud semántica |
| **Prompt template** | Define el contrato entre el sistema y el LLM |
| **RetrievalQA** | Orquesta todo el pipeline RAG con LangChain |
| **Gemini** | Genera la respuesta final fundamentada en el contexto |

### Lo que construiste en estos dos encuentros

Arrancaste entendiendo cómo un LLM lee texto (tokens → embeddings) y terminaste construyendo un sistema que responde preguntas sobre tus propios documentos. Ese camino — del concepto al sistema funcional — es exactamente lo que hacen los equipos de IA en producción.

### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas, para vos:

1. ¿Qué concepto de hoy te costó más encajar en la cabeza? ¿Por qué creés que se resistió?
2. Si tuvieras que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirías?

No hay respuesta correcta. Lo que escribás es el mapa de tu propio recorrido.